In [ ]:
%pip install "monai[all]"

In [ ]:
import torch

print(f"Is CUDA available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("PyTorch is currently using the CPU.")

In [ ]:
from monai.utils import first, set_determinism
from monai.transforms import (
    AsDiscrete,
    AsDiscreted,
    EnsureChannelFirstd,
    Compose,
    CropForegroundd,
    LoadImaged,
    Orientationd,
    RandCropByPosNegLabeld,
    SaveImaged,
    ScaleIntensityRanged,
    Spacingd,
    Invertd,
    RandAffined
)
from monai.handlers.utils import from_engine
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.metrics import DiceMetric
from monai.losses import DiceLoss
from monai.inferers import sliding_window_inference
from monai.data import CacheDataset, DataLoader, Dataset, decollate_batch
from monai.config import print_config
from monai.apps import download_and_extract
import torch
import matplotlib.pyplot as plt
import tempfile
import shutil
import os
import glob
import numpy as np
import random

print_config()

In [ ]:
data_dir = r"/kaggle/input/datasets/aakarroy17/tooth-fairy/Dataset112_ToothFairy2"

train_images = sorted(glob.glob(os.path.join(data_dir, "imagesTr", "*.mha")))
train_labels = sorted(glob.glob(os.path.join(data_dir, "labelsTr", "*.mha")))
data_dicts = [{"image": image_name, "label": label_name} for image_name, label_name in zip(train_images, train_labels)]

set_determinism(seed=0)
random.seed(0)

random.shuffle(data_dicts)

total_patients = len(data_dicts)
train_end = int(total_patients * 0.80)
val_end = int(total_patients * 0.90)

train_files = data_dicts[:train_end]
val_files = data_dicts[train_end:val_end]
test_files = data_dicts[val_end:] 

print("-" * 30)
print(f"Total Patients Found: {total_patients}")
print(f"Training on: {len(train_files)} patients (80%)")
print(f"Validating on: {len(val_files)} patients (10%)")
print(f"Blind Testing on: {len(test_files)} patients (10%)")
print("-" * 30)

In [ ]:
from monai.transforms import MapLabelValued

orig_labels = [
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 40, 
    11, 12, 13, 14, 15, 16, 17, 18,
    21, 22, 23, 24, 25, 26, 27, 28,
    31, 32, 33, 34, 35, 36, 37, 38,
    41, 42, 43, 44, 45, 46, 47, 48
]

new_labels = [
    1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
    0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0
]

In [ ]:
train_transforms = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        MapLabelValued(
            keys=["label"], 
            orig_labels=orig_labels, 
            target_labels=new_labels
        ),
        ScaleIntensityRanged(
            keys=["image"],
            a_min=200,
            a_max=2000,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        CropForegroundd(keys=["image", "label"], source_key="image", allow_smaller=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=(0.4, 0.4, 0.4), mode=("bilinear", "nearest")),
        RandCropByPosNegLabeld(
            keys=["image", "label"],
            label_key="label",
            spatial_size=(96, 96, 96),
            pos=1,
            neg=1,
            num_samples=2,
            image_key="image",
            image_threshold=0,
        ),
        RandAffined(
            keys=['image', 'label'],
            mode=('bilinear', 'nearest'),
            prob=0.5, spatial_size=(96, 96, 96),
            rotate_range=(0, 0, np.pi/15),
            scale_range=(0.1, 0.1, 0.1)),
    ]
)
val_transforms = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        MapLabelValued(
            keys=["label"], 
            orig_labels=orig_labels, 
            target_labels=new_labels
        ),
        ScaleIntensityRanged(
            keys=["image"],
            a_min=200,
            a_max=2000,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        CropForegroundd(keys=["image", "label"], source_key="image", allow_smaller=True),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=(0.4, 0.4, 0.4), mode=("bilinear", "nearest")),
    ]
)

In [ ]:
train_ds = Dataset(data=train_files, transform=train_transforms)

train_loader = DataLoader(
    train_ds, 
    batch_size=2, 
    shuffle=True, 
    num_workers=4,
    pin_memory=torch.cuda.is_available()
)

val_ds = Dataset(data=val_files, transform=val_transforms)

val_loader = DataLoader(
    val_ds, 
    batch_size=1, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=torch.cuda.is_available()
)

In [ ]:
root_dir = "/kaggle/working/MONAi_01"
os.makedirs(root_dir,exist_ok = True)

In [ ]:
TOTAL_CLASSES = 3
device = torch.device("cuda:0")
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=TOTAL_CLASSES,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.INSTANCE,
).to(device)
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4)
dice_metric = DiceMetric(include_background=False, reduction="mean")

In [ ]:
max_epochs = 20
val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
post_pred = Compose([AsDiscrete(argmax=True, to_onehot=TOTAL_CLASSES)])
post_label = Compose([AsDiscrete(to_onehot=TOTAL_CLASSES)])

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = (
            batch_data["image"].to(device),
            batch_data["label"].to(device),
        )
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        print(f"{step}/{len(train_ds) // train_loader.batch_size}, " f"train_loss: {loss.item():.4f}")
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = (
                    val_data["image"].to(device),
                    val_data["label"].to(device),
                )
                roi_size = (96, 96, 96)
                sw_batch_size = 4
                val_outputs = sliding_window_inference(val_inputs, roi_size, sw_batch_size, model)
                val_outputs = [post_pred(i) for i in decollate_batch(val_outputs)]
                val_labels = [post_label(i) for i in decollate_batch(val_labels)]
                dice_metric(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            dice_metric.reset()

            metric_values.append(metric)
            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), os.path.join(root_dir, "best_metric_model.pth"))
                print("saved new best metric model")
            print(
                f"current epoch: {epoch + 1} current mean dice: {metric:.4f}"
                f"\nbest mean dice: {best_metric:.4f} "
                f"at epoch: {best_metric_epoch}"
            )

In [ ]:
print(f"train completed, best_metric: {best_metric:.4f} " f"at epoch: {best_metric_epoch}")

In [ ]:
plt.figure("train", (12, 6))

plt.subplot(1, 2, 1)
plt.title("Epoch Average Loss")
x = [i + 1 for i in range(len(epoch_loss_values))]
y = epoch_loss_values
plt.xlabel("epoch")
plt.plot(x, y)

plt.subplot(1, 2, 2)
plt.title("Val Mean Dice")
x = [val_interval * (i + 1) for i in range(len(metric_values))]
y = metric_values
plt.xlabel("epoch")
plt.plot(x, y)

save_path = os.path.join(root_dir, "training_metrics.png")
plt.savefig(save_path, bbox_inches="tight", dpi=300)
print(f"Plot successfully saved to: {save_path}")

plt.show()

In [ ]:
model.load_state_dict(torch.load(os.path.join(root_dir, "best_metric_model.pth"), weights_only=True))
model.eval()
with torch.no_grad():
    for i, val_data in enumerate(val_loader):
        roi_size = (96, 96, 96)
        sw_batch_size = 4
        val_outputs = sliding_window_inference(val_data["image"].to(device), roi_size, sw_batch_size, model)
        plt.figure("check", (18, 6))
        plt.subplot(1, 3, 1)
        plt.title(f"image {i}")
        plt.imshow(val_data["image"][0, 0, :, :, 80], cmap="gray")
        plt.subplot(1, 3, 2)
        plt.title(f"label {i}")
        plt.imshow(val_data["label"][0, 0, :, :, 80])
        plt.subplot(1, 3, 3)
        plt.title(f"output {i}")
        plt.imshow(torch.argmax(val_outputs, dim=1).detach().cpu()[0, :, :, 80])
        plt.show()
        if i == 2:
            break

In [ ]:
test_org_transforms = Compose(
    [
        LoadImaged(keys="image"),
        EnsureChannelFirstd(keys="image"),
        Orientationd(keys=["image"], axcodes="RAS"),
        Spacingd(keys=["image"], pixdim=(0.4,0.4,0.4), mode="bilinear"),
        ScaleIntensityRanged(
            keys=["image"],
            a_min=200,
            a_max=2000,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        CropForegroundd(keys=["image"], source_key="image", allow_smaller=True),
    ]
)

test_org_ds = Dataset(data=test_files, transform=test_org_transforms)

test_org_loader = DataLoader(test_org_ds, batch_size=1, num_workers=0)

post_transforms = Compose(
    [
        AsDiscreted(keys="pred", argmax=True),
        
        Invertd(
            keys="pred",
            transform=test_org_transforms,
            orig_keys="image",
            meta_keys="pred_meta_dict",
            orig_meta_keys="image_meta_dict",
            meta_key_postfix="meta_dict",
            nearest_interp=True,
            to_tensor=True,
        ),
        
        SaveImaged(
            keys="pred", 
            meta_keys="pred_meta_dict", 
            output_dir=os.path.join(root_dir, "predictions"), 
            output_postfix="seg", 
            resample=False
        ),
    ]
)

In [ ]:
from monai.transforms import LoadImage
loader = LoadImage()

In [ ]:
model.load_state_dict(torch.load(os.path.join(root_dir, "best_metric_model.pth"), weights_only=True))
model.eval()

with torch.no_grad():
    for i, test_data in enumerate(test_org_loader):
        test_inputs = test_data["image"].to(device)
        roi_size = (96, 96, 96)
        sw_batch_size = 4
        test_data["pred"] = sliding_window_inference(test_inputs, roi_size, sw_batch_size, model)

        test_data = [post_transforms(item) for item in decollate_batch(test_data)]

        test_output = from_engine(["pred"])(test_data)

        original_image = loader(test_output[0].meta["filename_or_obj"])

        plt.figure("check", (18, 6))
        
        plt.subplot(1, 2, 1)
        plt.title(f"Patient {i+1} - Original Scan")
        plt.imshow(original_image[:, :, 20], cmap="gray")
        
        plt.subplot(1, 2, 2)
        plt.title(f"Patient {i+1} - Model Prediction")
        plt.imshow(test_output[0].detach().cpu()[0, :, :, 20]) 
        
        save_path = os.path.join(root_dir, f"inference_visualization_{i+1}.png")
        plt.savefig(save_path, bbox_inches="tight", dpi=300)
        print(f"Visualization perfectly saved to: {save_path}")

        plt.show()

In [ ]:
!zip -r Jaw-Separation /kaggle/working/MONAi_01